Here I'm going to make an optimization of all previous stuff.

In [17]:
import torch
import torch.nn.functional as F

In [18]:
words = open('names.txt', 'r').read().splitlines()
len(words), words[:8]

(32033,
 ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'])

In [19]:
chars = sorted(list(set(''.join(words))))
stoi = {s : i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [20]:
# build the dataset
block_size = 3 # how many characters do we take to predict the next one

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

X_train, y_train = build_dataset(words[:n1])
X_dev,   y_dev   = build_dataset(words[n1:n2])
X_test,  y_test  = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [21]:
X_train

tensor([[ 0,  0,  0],
        [ 0,  0, 25],
        [ 0, 25, 21],
        ...,
        [15, 12,  4],
        [12,  4,  1],
        [ 4,  1, 14]])

In [22]:
# MLP revisited
n_embedded_dim = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 

gen = torch.Generator().manual_seed(2147483647)
C =  torch.randn((vocab_size, n_embedded_dim),            generator=gen)
W1 = torch.randn((n_embedded_dim * block_size, n_hidden), generator=gen)
b1 = torch.randn(n_hidden,                                generator=gen)
W2 = torch.randn((n_hidden, vocab_size),                  generator=gen)
b2 = torch.randn(vocab_size,                              generator=gen)

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

11897


In [23]:
max_steps = 200_000
batch_size = 32
lossi = []
gen = torch.Generator().manual_seed(2147483647)

for i in range(max_steps):

    # minibatch constructor
    ix = torch.randint(0, X_train.shape[0], (batch_size,), generator=gen)
    Xb, Yb = X_train[ix], y_train[ix]

    # forward pass
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    hidden_preact = embcat @ W1 + b1
    hidden = torch.tanh(hidden_preact)
    logits = hidden @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if i < 100_000 else 0.01
    for p in parameters:
        p.data -= lr * p.grad

# emb.shape, hidden.shape, loss.item(), logits.shape, logits

In [24]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train' : (X_train, y_train),
        'val'   : (X_dev, y_dev),
        'test'  : (X_test, y_test)
    }[split]

    emb = C[x] # (N, block_size, n_embedded_dim)
    embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
    h = torch.tanh(embcat @ W1 + b1) # (N, n_hidden)
    logits = h @ W2 + b2 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')    

train 2.1251766681671143
val 2.1697797775268555


In [27]:
# sample from the model
g = torch.Generator().manual_seed(52)

for _ in range(20):
    
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])] # (1, block_size, n_embedded_dim)
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    
    print(''.join(itos[i] for i in out))

slean.
carliah.
hux.
elia.
blessa.
maranzo.
sten.
julisly.
did.
laulyn.
amari.
karli.
zhan.
dhettaden.
augadvi.
kaiden.
rhyna.
zachine.
stin.
thehlanne.
